0. SETUP


In [1]:
!pip install pypdf

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

!pip install rank_bm25

!pip install -q transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 8.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [52]:
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.0 MB/s eta 0:00:00


Model Setup

In [53]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


model_name_v3 = "Qwen/Qwen2.5-7B-Instruct"


# 4-bit quantization so the 7B model fits comfortably on Colab GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)


tokenizer_v3 = AutoTokenizer.from_pretrained(
    model_name_v3
)


generation_model_v3 = AutoModelForCausalLM.from_pretrained(
    model_name_v3,
    quantization_config=quantization_config,
    device_map="auto"
)


print("Loaded:", model_name_v3)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
checkpoint = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(checkpoint)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

generation_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(generation_model_name)

generation_model = AutoModelForCausalLM.from_pretrained(
    generation_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
from sentence_transformers import CrossEncoder

checkpoint_2 = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(checkpoint_2)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [49]:
def generation(
    query,
    context_items,
    generation_model,
    tokenizer
):
    if not context_items:
        return "I do not know based on the provided documents."

    context_parts = []

    for item in context_items:
        context_parts.append(
            f"""
Document: {item["document"]}
Page: {item["page"]}
Text:
{item["text"]}
"""
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a question-answering assistant.

Answer the question using only the information explicitly supported by the provided context.

Rules:
1. Use only information that is directly supported by the context.
2. Do not add explanations, assumptions, or facts from your own knowledge.
3. Include all important information from the context that is relevant to the question.
4. Do not combine separate pieces of information in a way that changes their meaning.
5. If the context contains an equation or formula, reproduce it only if it is explicitly shown in the context.
6. Do not create, derive, reconstruct, or modify equations or formulas.
7. If different passages describe related but different concepts, keep them separate.
8. If the context does not contain enough information to answer the question, say:
   "I do not know based on the provided documents."
9. Keep the answer concise and directly focused on the question.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(generation_model.device)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

Indexing Functions

In [6]:
from pypdf import PdfReader

doc_list = ["/content/1706.03762v7.pdf","/content/2405.07987v5.pdf","/content/NeurIPS_ML4PS_2025_87.pdf"]

def load_document(documents_list):
  text = []
  for doc in documents_list:
    reader = PdfReader(doc)
    for i,page in enumerate(reader.pages):
     text.append( {
          "document":doc,
          "page" : i+1,
          "text":page.extract_text()
      })
  return text



In [7]:
import re

def cleaning(documents_list):
  cleaned_results = []
  for doc in documents_list:
    text = doc["text"]
    if text is None:
      continue
    text = text.strip()
    if text == "":
      continue
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce 3+ newlines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    cleaned_results.append({
            "document": doc["document"],
            "page": doc["page"],
            "text": text
        })

  return cleaned_results



In [8]:
from nltk.tokenize import sent_tokenize

def chunk_documents_sentences(document_list, chunk_size=1000):
    results = []

    for doc in document_list:
        page_text = doc["text"]
        sentences = sent_tokenize(page_text)

        current_chunk = []
        chunk_id = 0

        for sentence in sentences:

            # If chunk is empty, add the sentence first
            if not current_chunk:
                current_chunk.append(sentence)
                continue

            current_text = " ".join(current_chunk)

            if len(current_text) + len(sentence) + 1 <= chunk_size:
                current_chunk.append(sentence)

            else:
                results.append({
                    "document": doc["document"],
                    "page": doc["page"],
                    "chunk_id": chunk_id,
                    "chunk_text": " ".join(current_chunk)
                })

                chunk_id += 1

                last_sentence = current_chunk[-1]

                current_chunk = [last_sentence, sentence]

        # Save final chunk
        if current_chunk:
            results.append({
                "document": doc["document"],
                "page": doc["page"],
                "chunk_id": chunk_id,
                "chunk_text": " ".join(current_chunk)
            })

    return results


In [9]:
load = load_document(doc_list)
clean = cleaning(load)
chunks = chunk_documents_sentences(clean, chunk_size=1000)
print(chunks)

[{'document': '/content/1706.03762v7.pdf', 'page': 1, 'chunk_id': 0, 'chunk_text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works. Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture

In [10]:
# def chunk_documents(cleaned_documents, chunk_size=1000, overlap=200):
#   chunk = []
#   for doc in cleaned_documents:
#     text = doc["text"]
#     start = 0
#     chunk_id = 0
#     while start < len(text):
#       end = start+chunk_size
#       chunk_text = text[start:end]

#       chunk.append(
#           {
#             "document":doc["document"],
#             "page":doc["page"],
#             "chunk_id":chunk_id,
#             "chunk_text":chunk_text
#           }
#       )
#       chunk_id +=1
#       start = start + (chunk_size-overlap)
#   return chunk



In [11]:
# from transformers import AutoTokenizer

# def token_based_chuncking(cleaned_doc,checkpoint,token_size=200, overlap=30):
#   tokenizer = AutoTokenizer.from_pretrained(checkpoint)
#   results = []
#   for doc in cleaned_doc:
#     text = doc["text"]
#     start = 0
#     chunk_id = 0
#     token_ids  = tokenizer.encode(text)
#     while start< len(token_ids) :
#       tokenized = token_ids[start:start+token_size]
#       decoded_text = tokenizer.decode(tokenized)
#       results.append({
#             "document":doc["document"],
#             "page":doc["page"],
#             "chunk_id":chunk_id,
#             "chunk_text":decoded_text})
#       start += token_size - overlap
#       chunk_id+=1
#   return results


In [12]:
def create_embeddings(model,active_corpus):
  chunk_list = []
  for chunk in active_corpus:
    chunk_list.append(chunk["chunk_text"])

  vectors = model.encode(chunk_list)
  return vectors



In [13]:
import numpy as np

from rank_bm25 import BM25Okapi

def build_bm25_index(active_corpus):
    tokenized_corpus = []

    for result in active_corpus:
        text = result["chunk_text"]
        text = text.lower()
        text = text.split()
        tokenized_corpus.append(text)

    bm25_index = BM25Okapi(tokenized_corpus)

    return bm25_index

In [14]:
from nltk.tokenize import sent_tokenize

def create_child_chunks(active_corpus, child_size=300):
    child_chunks = []

    for parent_index, parent in enumerate(active_corpus):

        parent_text = parent["chunk_text"]
        sentences = sent_tokenize(parent_text)

        current_chunk = []

        for sentence in sentences:

            if not current_chunk:
                current_chunk.append(sentence)
                continue

            current_text = " ".join(current_chunk)

            if len(current_text) + len(sentence) + 1 <= child_size:
                current_chunk.append(sentence)

            else:
                child_chunks.append({
                    "child_text": " ".join(current_chunk),
                    "parent_index": parent_index,
                    "document": parent["document"],
                    "page": parent["page"]
                })

                current_chunk = [sentence]

        # save the final child
        if current_chunk:
            child_chunks.append({
                "child_text": " ".join(current_chunk),
                "parent_index": parent_index,
                "document": parent["document"],
                "page": parent["page"]
            })

    return child_chunks

RETRIEVAL FUNCTIONS

In [15]:
import numpy as np
def compute_cos_similarity(query_vector,document_vector):
  return np.dot(document_vector,query_vector)/(np.linalg.norm(query_vector) *np.linalg.norm(document_vector,axis=1))

In [16]:
def retrieve_top_k(queries,embedded_vectors,model,top_k=20):
  ranked_indicies = []
  for query in queries:
    query_vector = model.encode(query)
    cos_similarity = compute_cos_similarity(query_vector,embedded_vectors)
    sorted_indices = np.argsort(cos_similarity)[::-1]
    top_indices = sorted_indices[:top_k]
    ranked_indicies.append(top_indices)


  return ranked_indicies



In [17]:
def bm25_retrival(queries,bm25_index,top_k=20):
  ranked_indicies = []
  for query in queries:
    query = query.lower()
    query = query.split()
    doc_scores = bm25_index.get_scores(query)
    sorted_list = np.argsort(doc_scores)[::-1]
    sorted_list = sorted_list[:top_k]

    ranked_indicies.append(sorted_list)

  return ranked_indicies


In [18]:
def hybrid_search(rankings, k=60, top_k=20):
    rrf_scores = {}

    for ranking in rankings:
        for rank, chunk_index in enumerate(ranking, start=1):
            chunk_index = int(chunk_index)

            if chunk_index in rrf_scores:
                rrf_scores[chunk_index] += 1 / (k + rank)
            else:
                rrf_scores[chunk_index] = 1 / (k + rank)

    ranked_results = sorted(
        rrf_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return ranked_results[:top_k]

In [19]:
def reranking(query, active_corpus, hybrid_results,reranker, top_k=5):

    cross_encoder_list = []
    candidate_indices = []

    for result in hybrid_results:
        index = result[0]

        candidate_indices.append(index)

        text = active_corpus[index]["chunk_text"]

        cross_encoder_list.append(
            [query, text]
        )

    scores = reranker.predict(cross_encoder_list)

    sorted_positions = scores.argsort()[::-1]

    reranked_results = []

    for position in sorted_positions[:top_k]:
        chunk_index = candidate_indices[position]
        reranker_score = scores[position]

        reranked_results.append(
            (chunk_index, reranker_score)
        )

    return reranked_results

 ADVANCED QUERY FUNCTIONS


In [20]:
import ast
def query_transformation(original_query,conversation_context, generation_model, tokenizer):

    prompt = f"""
You are an AI assistant tasked with reformulating user queries to improve retrieval in a RAG system based on the conversation context provided.

Rewrite this question into 3 to 5 different queries for document retrieval in a python list.

Conversation Context:
{conversation_context}

Original query:
{original_query}

Preserve the original meaning.
Do not answer the question.
Do not introduce new facts.
Return only the list of rewritten queries
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(generation_model.device)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    rewritten_query = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    queries = []
    for line in rewritten_query.splitlines():
      line = re.sub(r"^\s*\d+[\.\)]\s*", "", line).strip()

      if line:
          queries.append(line)

    return queries



In [21]:
def query_decomposition(original_query,conversation_context, generation_model, tokenizer):
  prompt = f"""
  Break the user query into independent sub-questions
only if necessary.

Conversation Context:
{conversation_context}

Original query:
{original_query}

Each sub-question should retrieve one distinct piece of evidence.

Preserve the user's intent.
Do not answer the questions.
Do not introduce new facts.
Return only the sub-questions."""

  messages = [
          {
              "role": "user",
              "content": prompt
          }
      ]

  formatted_prompt = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True
      )
  inputs = tokenizer(
          formatted_prompt,
          return_tensors="pt"
      ).to(generation_model.device)

  with torch.no_grad():
          outputs = generation_model.generate(
              **inputs,
              max_new_tokens=80,
              do_sample=False
          )

  generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

  rewritten_query = tokenizer.decode(
          generated_tokens,
          skip_special_tokens=True
      ).strip()

  queries = []
  for line in rewritten_query.splitlines():
        line = re.sub(r"^\s*\d+[\.\)]\s*", "", line).strip()

        if line:
            queries.append(line)

  return queries

In [22]:
def hyde(original_query,conversation_context,generation_model,tokenizer,embedding_model):
  prompt = f"""
  Write a short passage that would likely appear in a document
answering this query.
Conversation Context:
{conversation_context}

Original query:
{original_query}


Do not mention that it is hypothetical.
Do not explain the task.
Return only the passage. """

  messages = [
          {
              "role": "user",
              "content": prompt
          }
      ]

  formatted_prompt = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True
      )
  inputs = tokenizer(
          formatted_prompt,
          return_tensors="pt"
      ).to(generation_model.device)

  with torch.no_grad():
          outputs = generation_model.generate(
              **inputs,
              max_new_tokens=80,
              do_sample=False
          )

  generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

  generated_text = tokenizer.decode(
          generated_tokens,
          skip_special_tokens=True
      ).strip()

  vector = embedding_model.encode(generated_text)
  return vector,generated_text

def hyde_retreival (hyde_vector,corpus_embeddings,top_k):
  cos_similarity = compute_cos_similarity(hyde_vector,corpus_embeddings)
  sorted_indices = np.argsort(cos_similarity)[::-1]
  top_indices = sorted_indices[:top_k]
  return top_indices




In [23]:
def filter_by_metadata(active_corpus, document=None, page=None):
    filtered_corpus = []
    original_indices = []

    for i, chunk in enumerate(active_corpus):
        match = True

        if document is not None and chunk["document"] != document:
            match = False

        if page is not None and chunk["page"] != page:
            match = False

        if match:
            filtered_corpus.append(chunk)
            original_indices.append(i)

    return filtered_corpus, original_indices

ADVANCED CONTEXT FUNCTIONS

In [24]:
def parent_child_retrieval(
    query,
    child_corpus,
    child_embeddings,
    embedding_model,
    top_k=5
):
    query_vector = embedding_model.encode(query)

    scores = compute_cos_similarity(
        query_vector,
        child_embeddings
    )

    top_child_indices = np.argsort(scores)[::-1][:top_k]

    parent_indices = []
    seen_parents = set()

    for child_index in top_child_indices:
        parent_index = child_corpus[child_index]["parent_index"]

        if parent_index not in seen_parents:
            parent_indices.append(parent_index)
            seen_parents.add(parent_index)

    return parent_indices

In [25]:
def context_compression(
    parent_indices,
    query,
    active_corpus,
    embedding_model,
    top_k=5
):
    candidates = []

    for parent_index in parent_indices:
        parent = active_corpus[parent_index]

        sentences = sent_tokenize(
            parent["chunk_text"]
        )

        for sentence in sentences:
            candidates.append({
                "text": sentence,
                "parent_index": parent_index,
                "document": parent["document"],
                "page": parent["page"]
            })

    if not candidates:
        return []

    sentence_texts = [
        candidate["text"]
        for candidate in candidates
    ]

    sentence_embeddings = embedding_model.encode(
        sentence_texts
    )

    query_vector = embedding_model.encode(query)

    similarity = compute_cos_similarity(
        query_vector,
        sentence_embeddings
    )

    top_positions = np.argsort(similarity)[::-1][:top_k]

    compressed_context = []

    for position in top_positions:
        item = candidates[position].copy()
        item["score"] = float(similarity[position])

        compressed_context.append(item)

    return compressed_context

In [26]:
def context_from_indices(active_corpus, indices):
    context_items = []

    for index in indices:
        context_items.append({
            "text": active_corpus[index]["chunk_text"],
            "document": active_corpus[index]["document"],
            "page": active_corpus[index]["page"]
        })

    return context_items

INDEX BUILDING

In [28]:
# =========================================================
# INDEX BUILDING - RUN ONCE
# =========================================================

# Load documents
doc = load_document(doc_list)

# Clean documents
doc_cleaned = cleaning(doc)

# Build parent corpus
active_corpus = chunk_documents_sentences(
    doc_cleaned,
    chunk_size=1000
)

print("Parent chunks:", len(active_corpus))


# Create parent embeddings
corpus_embeddings = create_embeddings(
    model,
    active_corpus
)

print("Parent embedding shape:", corpus_embeddings.shape)


# Build BM25 index over complete corpus
bm25_index = build_bm25_index(
    active_corpus
)


# Build child corpus for parent-child retrieval
child_corpus = create_child_chunks(
    active_corpus,
    child_size=300
)

print("Child chunks:", len(child_corpus))


# Create child embeddings
child_texts = [
    child["child_text"]
    for child in child_corpus
]

child_embeddings = model.encode(
    child_texts
)

print("Child embedding shape:", child_embeddings.shape)

Parent chunks: 240
Parent embedding shape: (240, 384)
Child chunks: 860
Child embedding shape: (860, 384)


 QUERY-TIME PIPELINE

In [ ]:
# =========================================================
# QUERY-TIME PIPELINE
# =========================================================

conversation_context = (
    "We are discussing how the Transformer decoder works."
)

original_query = """
How does the Transformer decoder generate output,
why is masking necessary,
and how does it differ from the encoder?
"""


# =========================================================
# 1. QUERY DECOMPOSITION
# =========================================================

retrieval_queries = query_decomposition(
    original_query,
    conversation_context,
    generation_model,
    tokenizer
)

print("Retrieval queries:")

for q in retrieval_queries:
    print("-", q)


# =========================================================
# 2. OPTIONAL METADATA FILTER
# =========================================================

target_document = "/content/1706.03762v7.pdf"

if target_document is not None:

    search_corpus, original_indices = filter_by_metadata(
        active_corpus,
        document=target_document
    )

    search_embeddings = corpus_embeddings[
        original_indices
    ]

    search_bm25_index = build_bm25_index(
        search_corpus
    )

else:

    search_corpus = active_corpus

    original_indices = np.arange(
        len(active_corpus)
    )

    search_embeddings = corpus_embeddings

    search_bm25_index = bm25_index


print("Search chunks:", len(search_corpus))


# =========================================================
# 3. DENSE RETRIEVAL
# =========================================================

dense_local_rankings = retrieve_top_k(
    retrieval_queries,
    search_embeddings,
    model,
    top_k=20
)

dense_rankings = []

for ranking in dense_local_rankings:

    converted = [
        int(original_indices[index])
        for index in ranking
    ]

    dense_rankings.append(converted)


# =========================================================
# 4. BM25 RETRIEVAL
# =========================================================

bm25_local_rankings = bm25_retrival(
    retrieval_queries,
    search_bm25_index,
    top_k=20
)

bm25_rankings = []

for ranking in bm25_local_rankings:

    converted = [
        int(original_indices[index])
        for index in ranking
    ]

    bm25_rankings.append(converted)


# =========================================================
# 5. COLLECT RANKINGS FOR RRF
# =========================================================

all_rankings = (
    dense_rankings
    +
    bm25_rankings
)


# =========================================================
# 6. OPTIONAL HYDE
# =========================================================

use_hyde = False

if use_hyde:

    hyde_vector, hypothetical_document = hyde(
        original_query,
        conversation_context,
        generation_model,
        tokenizer,
        model
    )

    hyde_local_indices = hyde_retreival(
        hyde_vector,
        search_embeddings,
        top_k=20
    )

    hyde_ranking = [
        int(original_indices[index])
        for index in hyde_local_indices
    ]

    all_rankings.append(
        hyde_ranking
    )


# =========================================================
# 7. RRF
# =========================================================

hybrid_results = hybrid_search(
    all_rankings,
    k=60,
    top_k=20
)


# =========================================================
# 8. RERANK
# =========================================================

reranked_results = reranking(
    original_query,
    active_corpus,
    hybrid_results,
    reranker,
    top_k=5
)

final_indices = [
    result[0]
    for result in reranked_results
]


# =========================================================
# 9. CREATE FINAL CONTEXT
# =========================================================

final_context = context_from_indices(
    active_corpus,
    final_indices
)


# =========================================================
# 10. GENERATION
# =========================================================

answer = generation(
    original_query,
    final_context,
    generation_model,
    tokenizer
)

print("\nFinal answer:")
print(answer)

Retrieval queries:
- How does the Transformer decoder generate output?
- Why is masking necessary in the Transformer decoder?
- How does the Transformer decoder differ from the encoder?
Search chunks: 53

Final answer:
The Transformer decoder generates output through a process involving multiple steps, including residual connections, layer normalization, and multi-head attention. Masking is necessary because it prevents positions from attending to subsequent positions, ensuring that predictions for position \(i\) can depend only on the known outputs at positions less than \(i\).

Compared to the encoder, the main differences lie in their architectures and functions:
- **Encoder**: Composed of a stack of \(N = 6\) identical layers, each containing two sub-layers. The first sub-layer is a multi-head self-attention mechanism, while the second is a simple, position-wise fully connected feed-forward network.
- **Decoder**: Also composed of a stack of \(N = 6\) identical layers, but differs 

In [ ]:
# =========================================================
# PARENT-CHILD + CONTEXT COMPRESSION TEST
# =========================================================

query = "Why is masking necessary in the Transformer decoder?"


# Retrieve small children → return large parents
parent_indices = parent_child_retrieval(
    query,
    child_corpus,
    child_embeddings,
    model,
    top_k=5
)

print("Retrieved parents:", parent_indices)


# Compress the parent context
compressed_context = context_compression(
    parent_indices,
    query,
    active_corpus,
    model,
    top_k=5
)


print("\nCompressed context:")

for item in compressed_context:
    print("Score:", item["score"])
    print("Document:", item["document"])
    print("Page:", item["page"])
    print("Text:", item["text"])
    print("-----")


# Generate using compressed context
compressed_answer = generation(
    query,
    compressed_context,
    generation_model,
    tokenizer
)

print("\nCompressed-context answer:")
print(compressed_answer)

Retrieved parents: [12, 93, 9, 7]

Compressed context:
Score: 0.512485921382904
Document: /content/1706.03762v7.pdf
Page: 3
Text: This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.
-----
Score: 0.4672187864780426
Document: /content/1706.03762v7.pdf
Page: 3
Text: Decoder: The decoder is also composed of a stack of N = 6identical layers.
-----
Score: 0.46523424983024597
Document: /content/2405.07987v5.pdf
Page: 6
Text: Masked Autoencoders (He et al., 2021) optimize randomly
sampled reconstruction tasks.
-----
Score: 0.44674810767173767
Document: /content/1706.03762v7.pdf
Page: 3
Text: In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack.
-----
Score: 0.4410628080368042
Document: /content/1706.03762v7.pdf
Page: 2
Text: To the best of

Evaluation Dataset

In [29]:
eval_dataset = [

    # ---------------- TRANSFORMER PAPER ----------------

    {
        "question": "Why is masking necessary in the Transformer decoder?",
        "type": "semantic",
        "relevant_chunks": [
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 3,
                "chunk_id": 1
            },
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 5,
                "chunk_id": 2
            }
        ],
        "expected_facts": [
            "decoder positions must not attend to future positions",
            "masking preserves autoregressive generation",
            "position i depends only on known earlier outputs"
        ],
        "should_abstain": False
    },

    {
        "question": "How does the Transformer decoder generate an output sequence?",
        "type": "semantic",
        "relevant_chunks": [
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 2,
                "chunk_id": 5
            },
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 2,
                "chunk_id": 6
            }
        ],
        "expected_facts": [
            "decoder generates symbols one element at a time",
            "generation is autoregressive",
            "previously generated symbols are used to generate the next"
        ],
        "should_abstain": False
    },

    {
        "question": "How does the Transformer decoder architecture differ from the encoder?",
        "type": "multi_part",
        "relevant_chunks": [
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 3,
                "chunk_id": 0
            },
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 3,
                "chunk_id": 1
            }
        ],
        "expected_facts": [
            "encoder layers contain self-attention and feed-forward sublayers",
            "decoder adds a third encoder-decoder attention sublayer",
            "decoder self-attention is masked"
        ],
        "should_abstain": False
    },

    {
        "question": "Why does scaled dot-product attention divide by the square root of the key dimension?",
        "type": "detail",
        "relevant_chunks": [
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 4,
                "chunk_id": 2
            }
        ],
        "expected_facts": [
            "large key dimensions produce large dot products",
            "large dot products push softmax toward very small gradients",
            "scaling by the square root of dk counteracts this"
        ],
        "should_abstain": False
    },

    {
        "question": "What advantage does multi-head attention provide over a single attention head?",
        "type": "semantic",
        "relevant_chunks": [
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 5,
                "chunk_id": 0
            }
        ],
        "expected_facts": [
            "multiple heads attend to different representation subspaces",
            "multiple heads can attend to different positions"
        ],
        "should_abstain": False
    },

    {
        "question": "Why does the Transformer need positional encoding and why did the authors choose sinusoidal encodings?",
        "type": "multi_part",
        "relevant_chunks": [
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 6,
                "chunk_id": 0
            },
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 6,
                "chunk_id": 1
            },
            {
                "document": "/content/1706.03762v7.pdf",
                "page": 6,
                "chunk_id": 2
            }
        ],
        "expected_facts": [
            "the model has no recurrence or convolution to represent order",
            "positional information is added to embeddings",
            "sinusoidal encodings may help learn relative positions",
            "they may extrapolate to longer sequence lengths"
        ],
        "should_abstain": False
    },


    # ---------------- PLATONIC REPRESENTATION PAPER ----------------

    {
        "question": "What is the Platonic Representation Hypothesis?",
        "type": "semantic",
        "relevant_chunks": [
            {
                "document": "/content/2405.07987v5.pdf",
                "page": 2,
                "chunk_id": 0
            },
            {
                "document": "/content/2405.07987v5.pdf",
                "page": 2,
                "chunk_id": 1
            }
        ],
        "expected_facts": [
            "different models may converge toward a shared representation",
            "representations model an underlying reality",
            "alignment is expected to increase with scale and information"
        ],
        "should_abstain": False
    },

    {
        "question": "How do the authors define and measure representational alignment?",
        "type": "detail",
        "relevant_chunks": [
            {
                "document": "/content/2405.07987v5.pdf",
                "page": 2,
                "chunk_id": 5
            },
            {
                "document": "/content/2405.07987v5.pdf",
                "page": 2,
                "chunk_id": 6
            }
        ],
        "expected_facts": [
            "alignment compares similarity structures induced by representations",
            "these similarity structures are represented as kernels",
            "the experiments use a mutual nearest-neighbor metric"
        ],
        "should_abstain": False
    },

    {
        "question": "What limitation prevents representations from different modalities from always converging perfectly?",
        "type": "semantic",
        "relevant_chunks": [
            {
                "document": "/content/2405.07987v5.pdf",
                "page": 10,
                "chunk_id": 1
            },
            {
                "document": "/content/2405.07987v5.pdf",
                "page": 10,
                "chunk_id": 2
            }
        ],
        "expected_facts": [
            "different modalities can contain fundamentally different information",
            "observations may be lossy or non-bijective",
            "alignment can be limited by available mutual information and model capacity"
        ],
        "should_abstain": False
    },


    # ---------------- ASTRONOMY PRH PAPER ----------------

    {
        "question": "Why is astronomy a useful testbed for the Platonic Representation Hypothesis?",
        "type": "semantic",
        "relevant_chunks": [
            {
                "document": "/content/NeurIPS_ML4PS_2025_87.pdf",
                "page": 2,
                "chunk_id": 3
            },
            {
                "document": "/content/NeurIPS_ML4PS_2025_87.pdf",
                "page": 2,
                "chunk_id": 4
            }
        ],
        "expected_facts": [
            "astronomical modalities are different projections of the same physical reality",
            "images spectra and photometry arise from shared physical processes",
            "this allows cross-modal representation convergence to be tested"
        ],
        "should_abstain": False
    },


    # ---------------- UNANSWERABLE ----------------

    {
        "question": "What was the exact financial cost in US dollars of training the Transformer big model?",
        "type": "unanswerable",
        "relevant_chunks": [],
        "expected_facts": [],
        "should_abstain": True
    },

    {
        "question": "According to these papers, what caused the 2008 global financial crisis?",
        "type": "unanswerable",
        "relevant_chunks": [],
        "expected_facts": [],
        "should_abstain": True
    }
]

In [30]:
def resolve_relevant_indices(eval_dataset, active_corpus):

    chunk_lookup = {}

    for index, chunk in enumerate(active_corpus):

        key = (
            chunk["document"],
            chunk["page"],
            chunk["chunk_id"]
        )

        chunk_lookup[key] = index


    for example in eval_dataset:

        relevant_indices = []

        for chunk in example["relevant_chunks"]:

            key = (
                chunk["document"],
                chunk["page"],
                chunk["chunk_id"]
            )

            if key not in chunk_lookup:
                raise ValueError(
                    f"Ground-truth chunk not found: {key}"
                )

            relevant_indices.append(
                chunk_lookup[key]
            )

        example["relevant_indices"] = relevant_indices

    return eval_dataset

In [31]:
eval_dataset = resolve_relevant_indices(
    eval_dataset,
    active_corpus
)

In [32]:
for example in eval_dataset:

    print(example["question"])
    print("Relevant:", example["relevant_indices"])
    print("Abstain:", example["should_abstain"])
    print("-----")

Why is masking necessary in the Transformer decoder?
Relevant: [12, 19]
Abstain: False
-----
How does the Transformer decoder generate an output sequence?
Relevant: [9, 10]
Abstain: False
-----
How does the Transformer decoder architecture differ from the encoder?
Relevant: [11, 12]
Abstain: False
-----
Why does scaled dot-product attention divide by the square root of the key dimension?
Relevant: [15]
Abstain: False
-----
What advantage does multi-head attention provide over a single attention head?
Relevant: [17]
Abstain: False
-----
Why does the Transformer need positional encoding and why did the authors choose sinusoidal encodings?
Relevant: [21, 22, 23]
Abstain: False
-----
What is the Platonic Representation Hypothesis?
Relevant: [58, 59]
Abstain: False
-----
How do the authors define and measure representational alignment?
Relevant: [63, 64]
Abstain: False
-----
What limitation prevents representations from different modalities from always converging perfectly?
Relevant: [117, 

In [33]:
def hit_at_k(retrieved_indices, relevant_indices, k):
    for index in retrieved_indices[:k]:
        if index in relevant_indices:
            return 1

    return 0

In [34]:
def recall_at_k(retrieved_indices, relevant_indices, k):
  count = 0
  for index in retrieved_indices[:k]:
    if index in relevant_indices:
      count+=1

  number = len(relevant_indices)
  if number == 0:
    return 0
  recall = count/number
  return recall


In [35]:
def reciprocal_rank(retrieved_indices, relevant_indices, k):
    for i, index in enumerate(retrieved_indices[:k]):
        i = i + 1
        if index in relevant_indices:
            mrr = 1 / i
            return mrr

    return 0

In [36]:
# =========================================================
# TEST ONE EVALUATION QUESTION
# =========================================================

example = eval_dataset[0]

original_query = example["question"]
relevant_indices = example["relevant_indices"]

conversation_context = ""

print("Question:")
print(original_query)

print("\nGround-truth relevant indices:")
print(relevant_indices)


# =========================================================
# 1. QUERY DECOMPOSITION
# =========================================================

retrieval_queries = query_decomposition(
    original_query,
    conversation_context,
    generation_model,
    tokenizer
)

print("\nRetrieval queries:")

for q in retrieval_queries:
    print("-", q)


# =========================================================
# 2. OPTIONAL METADATA FILTER
# =========================================================

# Do not use the known ground-truth document during evaluation.
# That would leak information into retrieval.

target_document = None

if target_document is not None:

    search_corpus, original_indices = filter_by_metadata(
        active_corpus,
        document=target_document
    )

    search_embeddings = corpus_embeddings[
        original_indices
    ]

    search_bm25_index = build_bm25_index(
        search_corpus
    )

else:

    search_corpus = active_corpus

    original_indices = np.arange(
        len(active_corpus)
    )

    search_embeddings = corpus_embeddings

    search_bm25_index = bm25_index


print("\nSearch chunks:", len(search_corpus))


# =========================================================
# 3. DENSE RETRIEVAL
# =========================================================

dense_local_rankings = retrieve_top_k(
    retrieval_queries,
    search_embeddings,
    model,
    top_k=20
)

dense_rankings = []

for ranking in dense_local_rankings:

    converted = [
        int(original_indices[index])
        for index in ranking
    ]

    dense_rankings.append(converted)


# =========================================================
# 4. BM25 RETRIEVAL
# =========================================================

bm25_local_rankings = bm25_retrival(
    retrieval_queries,
    search_bm25_index,
    top_k=20
)

bm25_rankings = []

for ranking in bm25_local_rankings:

    converted = [
        int(original_indices[index])
        for index in ranking
    ]

    bm25_rankings.append(converted)


# =========================================================
# 5. COLLECT RANKINGS FOR RRF
# =========================================================

all_rankings = (
    dense_rankings
    +
    bm25_rankings
)


# =========================================================
# 6. OPTIONAL HYDE
# =========================================================

use_hyde = False

if use_hyde:

    hyde_vector, hypothetical_document = hyde(
        original_query,
        conversation_context,
        generation_model,
        tokenizer,
        model
    )

    hyde_local_indices = hyde_retreival(
        hyde_vector,
        search_embeddings,
        top_k=20
    )

    hyde_ranking = [
        int(original_indices[index])
        for index in hyde_local_indices
    ]

    all_rankings.append(
        hyde_ranking
    )


# =========================================================
# 7. RRF
# =========================================================

hybrid_results = hybrid_search(
    all_rankings,
    k=60,
    top_k=20
)

hybrid_indices = [
    result[0]
    for result in hybrid_results
]


# =========================================================
# 8. RERANK
# =========================================================

reranked_results = reranking(
    original_query,
    active_corpus,
    hybrid_results,
    reranker,
    top_k=5
)

final_indices = [
    result[0]
    for result in reranked_results
]


# =========================================================
# 9. HYBRID METRICS
# =========================================================

hybrid_hit_5 = hit_at_k(
    hybrid_indices,
    relevant_indices,
    k=5
)

hybrid_recall_5 = recall_at_k(
    hybrid_indices,
    relevant_indices,
    k=5
)

hybrid_rr_5 = reciprocal_rank(
    hybrid_indices,
    relevant_indices,
    k=5
)


# =========================================================
# 10. RERANKED METRICS
# =========================================================

reranked_hit_5 = hit_at_k(
    final_indices,
    relevant_indices,
    k=5
)

reranked_recall_5 = recall_at_k(
    final_indices,
    relevant_indices,
    k=5
)

reranked_rr_5 = reciprocal_rank(
    final_indices,
    relevant_indices,
    k=5
)


# =========================================================
# 11. RESULTS
# =========================================================

print("\n========================================")
print("EVALUATION RESULT")
print("========================================")

print("\nQuestion:")
print(original_query)

print("\nGround truth:")
print(relevant_indices)


print("\n----------------------------------------")
print("HYBRID RETRIEVAL")
print("----------------------------------------")

print("Top 5:")
print(hybrid_indices[:5])

print("Hit@5:")
print(hybrid_hit_5)

print("Recall@5:")
print(hybrid_recall_5)

print("Reciprocal Rank@5:")
print(hybrid_rr_5)


print("\n----------------------------------------")
print("RERANKED RETRIEVAL")
print("----------------------------------------")

print("Top 5:")
print(final_indices[:5])

print("Hit@5:")
print(reranked_hit_5)

print("Recall@5:")
print(reranked_recall_5)

print("Reciprocal Rank@5:")
print(reranked_rr_5)

Question:
Why is masking necessary in the Transformer decoder?

Ground-truth relevant indices:
[12, 19]

Retrieval queries:
- What is masking in the context of the Transformer decoder?
- Why is it important to mask parts of the input during training?

Search chunks: 240

EVALUATION RESULT

Question:
Why is masking necessary in the Transformer decoder?

Ground truth:
[12, 19]

----------------------------------------
HYBRID RETRIEVAL
----------------------------------------
Top 5:
[7, 19, 18, 52, 23]
Hit@5:
1
Recall@5:
0.5
Reciprocal Rank@5:
0.5

----------------------------------------
RERANKED RETRIEVAL
----------------------------------------
Top 5:
[19, 12, 18, 11, 7]
Hit@5:
1
Recall@5:
1.0
Reciprocal Rank@5:
1.0


In [37]:
# =========================================================
# EVALUATE ALL ANSWERABLE QUESTIONS
# =========================================================

evaluation_results = []

conversation_context = ""

for number, example in enumerate(eval_dataset, start=1):

    # Skip unanswerable questions for retrieval metrics
    if example["should_abstain"]:
        continue

    original_query = example["question"]
    relevant_indices = example["relevant_indices"]

    print("\n" + "=" * 70)
    print(f"QUESTION {number}")
    print("=" * 70)
    print(original_query)


    # =====================================================
    # 1. QUERY DECOMPOSITION
    # =====================================================

    retrieval_queries = query_decomposition(
        original_query,
        conversation_context,
        generation_model,
        tokenizer
    )


    # =====================================================
    # 2. SEARCH CORPUS
    # =====================================================

    # No metadata filtering during evaluation
    # because we do not want to leak the correct document.

    search_corpus = active_corpus

    original_indices = np.arange(
        len(active_corpus)
    )

    search_embeddings = corpus_embeddings

    search_bm25_index = bm25_index


    # =====================================================
    # 3. DENSE RETRIEVAL
    # =====================================================

    dense_local_rankings = retrieve_top_k(
        retrieval_queries,
        search_embeddings,
        model,
        top_k=20
    )

    dense_rankings = []

    for ranking in dense_local_rankings:

        converted = [
            int(original_indices[index])
            for index in ranking
        ]

        dense_rankings.append(converted)


    # =====================================================
    # 4. BM25 RETRIEVAL
    # =====================================================

    bm25_local_rankings = bm25_retrival(
        retrieval_queries,
        search_bm25_index,
        top_k=20
    )

    bm25_rankings = []

    for ranking in bm25_local_rankings:

        converted = [
            int(original_indices[index])
            for index in ranking
        ]

        bm25_rankings.append(converted)


    # =====================================================
    # 5. RRF
    # =====================================================

    all_rankings = (
        dense_rankings
        +
        bm25_rankings
    )

    hybrid_results = hybrid_search(
        all_rankings,
        k=60,
        top_k=20
    )

    hybrid_indices = [
        result[0]
        for result in hybrid_results
    ]


    # =====================================================
    # 6. RERANK
    # =====================================================

    reranked_results = reranking(
        original_query,
        active_corpus,
        hybrid_results,
        reranker,
        top_k=5
    )

    final_indices = [
        result[0]
        for result in reranked_results
    ]


    # =====================================================
    # 7. HYBRID METRICS
    # =====================================================

    hybrid_hit = hit_at_k(
        hybrid_indices,
        relevant_indices,
        k=5
    )

    hybrid_recall = recall_at_k(
        hybrid_indices,
        relevant_indices,
        k=5
    )

    hybrid_rr = reciprocal_rank(
        hybrid_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # 8. RERANKED METRICS
    # =====================================================

    reranked_hit = hit_at_k(
        final_indices,
        relevant_indices,
        k=5
    )

    reranked_recall = recall_at_k(
        final_indices,
        relevant_indices,
        k=5
    )

    reranked_rr = reciprocal_rank(
        final_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # 9. SAVE RESULTS
    # =====================================================

    evaluation_results.append({
        "question": original_query,

        "hybrid_hit@5": hybrid_hit,
        "hybrid_recall@5": hybrid_recall,
        "hybrid_rr@5": hybrid_rr,

        "reranked_hit@5": reranked_hit,
        "reranked_recall@5": reranked_recall,
        "reranked_rr@5": reranked_rr
    })


    # =====================================================
    # 10. PRINT QUESTION RESULT
    # =====================================================

    print("\nGround truth:")
    print(relevant_indices)

    print("\nHybrid Top 5:")
    print(hybrid_indices[:5])

    print(
        f"Hybrid -> "
        f"Hit@5: {hybrid_hit} | "
        f"Recall@5: {hybrid_recall:.3f} | "
        f"RR@5: {hybrid_rr:.3f}"
    )

    print("\nReranked Top 5:")
    print(final_indices)

    print(
        f"Reranked -> "
        f"Hit@5: {reranked_hit} | "
        f"Recall@5: {reranked_recall:.3f} | "
        f"RR@5: {reranked_rr:.3f}"
    )


# =========================================================
# OVERALL METRICS
# =========================================================

hybrid_hit_rate = np.mean([
    result["hybrid_hit@5"]
    for result in evaluation_results
])

hybrid_mean_recall = np.mean([
    result["hybrid_recall@5"]
    for result in evaluation_results
])

hybrid_mrr = np.mean([
    result["hybrid_rr@5"]
    for result in evaluation_results
])


reranked_hit_rate = np.mean([
    result["reranked_hit@5"]
    for result in evaluation_results
])

reranked_mean_recall = np.mean([
    result["reranked_recall@5"]
    for result in evaluation_results
])

reranked_mrr = np.mean([
    result["reranked_rr@5"]
    for result in evaluation_results
])


# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n\n" + "=" * 70)
print("OVERALL RETRIEVAL EVALUATION")
print("=" * 70)

print("\nHYBRID")
print(f"Hit Rate@5:    {hybrid_hit_rate:.3f}")
print(f"Mean Recall@5: {hybrid_mean_recall:.3f}")
print(f"MRR@5:         {hybrid_mrr:.3f}")

print("\nHYBRID + RERANKER")
print(f"Hit Rate@5:    {reranked_hit_rate:.3f}")
print(f"Mean Recall@5: {reranked_mean_recall:.3f}")
print(f"MRR@5:         {reranked_mrr:.3f}")


QUESTION 1
Why is masking necessary in the Transformer decoder?

Ground truth:
[12, 19]

Hybrid Top 5:
[7, 19, 18, 52, 23]
Hybrid -> Hit@5: 1 | Recall@5: 0.500 | RR@5: 0.500

Reranked Top 5:
[19, 12, 18, 11, 7]
Reranked -> Hit@5: 1 | Recall@5: 1.000 | RR@5: 1.000

QUESTION 2
How does the Transformer decoder generate an output sequence?

Ground truth:
[9, 10]

Hybrid Top 5:
[9, 10, 18, 11, 6]
Hybrid -> Hit@5: 1 | Recall@5: 1.000 | RR@5: 1.000

Reranked Top 5:
[10, 9, 18, 20, 12]
Reranked -> Hit@5: 1 | Recall@5: 1.000 | RR@5: 1.000

QUESTION 3
How does the Transformer decoder architecture differ from the encoder?

Ground truth:
[11, 12]

Hybrid Top 5:
[18, 11, 12, 9, 19]
Hybrid -> Hit@5: 1 | Recall@5: 1.000 | RR@5: 0.500

Reranked Top 5:
[11, 18, 9, 12, 39]
Reranked -> Hit@5: 1 | Recall@5: 1.000 | RR@5: 1.000

QUESTION 4
Why does scaled dot-product attention divide by the square root of the key dimension?

Ground truth:
[15]

Hybrid Top 5:
[13, 14, 19, 2, 35]
Hybrid -> Hit@5: 0 | Recall

In [38]:
print("\nRelevant chunk ranks:")

for relevant_index in relevant_indices:

    if relevant_index in hybrid_indices:
        hybrid_rank = hybrid_indices.index(relevant_index) + 1
    else:
        hybrid_rank = None

    if relevant_index in final_indices:
        reranked_rank = final_indices.index(relevant_index) + 1
    else:
        reranked_rank = None

    print(
        f"Chunk {relevant_index} | "
        f"Hybrid rank: {hybrid_rank} | "
        f"Reranked rank: {reranked_rank}"
    )


Relevant chunk ranks:
Chunk 200 | Hybrid rank: 16 | Reranked rank: 3
Chunk 201 | Hybrid rank: None | Reranked rank: None


In [40]:
# =========================================================
# DIAGNOSE QUESTION 9
# =========================================================

example = eval_dataset[8]   # Question 9

original_query = example["question"]
relevant_indices = example["relevant_indices"]

conversation_context = ""

print("Question:")
print(original_query)

print("\nGround-truth relevant indices:")
print(relevant_indices)


# =========================================================
# 1. QUERY DECOMPOSITION
# =========================================================

# retrieval_queries = query_decomposition(
#     original_query,
#     conversation_context,
#     generation_model,
#     tokenizer
# )

# print("\nRetrieval queries:")

# for q in retrieval_queries:
#     print("-", q)

# =========================================================
# 1. ORIGINAL QUERY ONLY — NO DECOMPOSITION
# =========================================================

retrieval_queries = [original_query]

print("\nRetrieval queries:")
for q in retrieval_queries:
    print("-", q)

# =========================================================
# 2. SEARCH CORPUS
# =========================================================

# No metadata filtering during evaluation

search_corpus = active_corpus

original_indices = np.arange(
    len(active_corpus)
)

search_embeddings = corpus_embeddings

search_bm25_index = bm25_index

print("\nSearch chunks:", len(search_corpus))


# =========================================================
# 3. DENSE RETRIEVAL
# =========================================================

dense_local_rankings = retrieve_top_k(
    retrieval_queries,
    search_embeddings,
    model,
    top_k=20
)

dense_rankings = []

for ranking in dense_local_rankings:

    converted = [
        int(original_indices[index])
        for index in ranking
    ]

    dense_rankings.append(converted)


# =========================================================
# 4. BM25 RETRIEVAL
# =========================================================

bm25_local_rankings = bm25_retrival(
    retrieval_queries,
    search_bm25_index,
    top_k=20
)

bm25_rankings = []

for ranking in bm25_local_rankings:

    converted = [
        int(original_indices[index])
        for index in ranking
    ]

    bm25_rankings.append(converted)


# =========================================================
# 5. COMBINE RANKINGS
# =========================================================

all_rankings = (
    dense_rankings
    +
    bm25_rankings
)


# =========================================================
# 6. OPTIONAL HYDE
# =========================================================

use_hyde = False

if use_hyde:

    hyde_vector, hypothetical_document = hyde(
        original_query,
        conversation_context,
        generation_model,
        tokenizer,
        model
    )

    hyde_local_indices = hyde_retreival(
        hyde_vector,
        search_embeddings,
        top_k=20
    )

    hyde_ranking = [
        int(original_indices[index])
        for index in hyde_local_indices
    ]

    all_rankings.append(
        hyde_ranking
    )


# =========================================================
# 7. RRF / HYBRID RETRIEVAL
# =========================================================

hybrid_results = hybrid_search(
    all_rankings,
    k=60,
    top_k=20
)

hybrid_indices = [
    result[0]
    for result in hybrid_results
]


# =========================================================
# 8. RERANK
# =========================================================

reranked_results = reranking(
    original_query,
    active_corpus,
    hybrid_results,
    reranker,
    top_k=5
)

final_indices = [
    result[0]
    for result in reranked_results
]


# =========================================================
# 9. METRICS
# =========================================================

hybrid_hit = hit_at_k(
    hybrid_indices,
    relevant_indices,
    k=5
)

hybrid_recall = recall_at_k(
    hybrid_indices,
    relevant_indices,
    k=5
)

hybrid_rr = reciprocal_rank(
    hybrid_indices,
    relevant_indices,
    k=5
)


reranked_hit = hit_at_k(
    final_indices,
    relevant_indices,
    k=5
)

reranked_recall = recall_at_k(
    final_indices,
    relevant_indices,
    k=5
)

reranked_rr = reciprocal_rank(
    final_indices,
    relevant_indices,
    k=5
)


# =========================================================
# 10. FIND RELEVANT CHUNK RANKS
# =========================================================

print("\n" + "=" * 70)
print("QUESTION 9 DIAGNOSTIC")
print("=" * 70)

print("\nQuestion:")
print(original_query)

print("\nGround truth:")
print(relevant_indices)


print("\n----------------------------------------")
print("DENSE RANKINGS")
print("----------------------------------------")

for i, ranking in enumerate(dense_rankings, start=1):
    print(f"Dense query {i}:")
    print(ranking[:20])


print("\n----------------------------------------")
print("BM25 RANKINGS")
print("----------------------------------------")

for i, ranking in enumerate(bm25_rankings, start=1):
    print(f"BM25 query {i}:")
    print(ranking[:20])


print("\n----------------------------------------")
print("HYBRID TOP 20")
print("----------------------------------------")

print(hybrid_indices)


print("\n----------------------------------------")
print("RERANKED TOP 5")
print("----------------------------------------")

print(final_indices)


print("\n----------------------------------------")
print("RELEVANT CHUNK RANKS")
print("----------------------------------------")

for relevant_index in relevant_indices:

    if relevant_index in hybrid_indices:
        hybrid_rank = hybrid_indices.index(relevant_index) + 1
    else:
        hybrid_rank = None

    if relevant_index in final_indices:
        reranked_rank = final_indices.index(relevant_index) + 1
    else:
        reranked_rank = None

    print(
        f"Chunk {relevant_index} | "
        f"Hybrid rank: {hybrid_rank} | "
        f"Reranked rank: {reranked_rank}"
    )


# =========================================================
# 11. CHECK WHETHER EACH RETRIEVER FOUND THE RELEVANT CHUNKS
# =========================================================

print("\n----------------------------------------")
print("WHERE WERE THE RELEVANT CHUNKS FOUND?")
print("----------------------------------------")

for relevant_index in relevant_indices:

    print(f"\nChunk {relevant_index}")

    for i, ranking in enumerate(dense_rankings, start=1):

        if relevant_index in ranking:
            rank = ranking.index(relevant_index) + 1
            print(
                f"  Dense query {i}: rank {rank}"
            )
        else:
            print(
                f"  Dense query {i}: not found"
            )

    for i, ranking in enumerate(bm25_rankings, start=1):

        if relevant_index in ranking:
            rank = ranking.index(relevant_index) + 1
            print(
                f"  BM25 query {i}: rank {rank}"
            )
        else:
            print(
                f"  BM25 query {i}: not found"
            )


# =========================================================
# 12. FINAL METRICS
# =========================================================

print("\n" + "=" * 70)
print("METRICS")
print("=" * 70)

print("\nHybrid")
print(f"Hit@5:    {hybrid_hit}")
print(f"Recall@5: {hybrid_recall:.3f}")
print(f"RR@5:     {hybrid_rr:.3f}")

print("\nHybrid + Reranker")
print(f"Hit@5:    {reranked_hit}")
print(f"Recall@5: {reranked_recall:.3f}")
print(f"RR@5:     {reranked_rr:.3f}")

Question:
What limitation prevents representations from different modalities from always converging perfectly?

Ground-truth relevant indices:
[117, 118]

Retrieval queries:
- What limitation prevents representations from different modalities from always converging perfectly?

Search chunks: 240

QUESTION 9 DIAGNOSTIC

Question:
What limitation prevents representations from different modalities from always converging perfectly?

Ground truth:
[117, 118]

----------------------------------------
DENSE RANKINGS
----------------------------------------
Dense query 1:
[85, 118, 107, 113, 125, 55, 56, 53, 76, 117, 68, 192, 61, 98, 176, 72, 60, 119, 93, 94]

----------------------------------------
BM25 RANKINGS
----------------------------------------
BM25 query 1:
[61, 198, 52, 76, 212, 62, 200, 113, 106, 65, 55, 98, 119, 197, 115, 99, 51, 181, 97, 214]

----------------------------------------
HYBRID TOP 20
----------------------------------------
[113, 76, 61, 55, 98, 119, 85, 118, 198, 

In [41]:
# =========================================================
# RETRIEVAL ABLATION
#
# A. Dense only
# B. BM25 only
# C. Dense + BM25 + RRF
# D. Dense + BM25 + RRF + Reranker
#
# Original query only:
# - NO query decomposition
# - NO HyDE
# - NO metadata filtering
# =========================================================


ablation_results = []


# =========================================================
# LOOP THROUGH EVALUATION DATASET
# =========================================================

for number, example in enumerate(eval_dataset, start=1):

    # -----------------------------------------------------
    # Skip unanswerable questions for retrieval evaluation
    # -----------------------------------------------------

    if example["should_abstain"]:
        continue


    original_query = example["question"]
    relevant_indices = example["relevant_indices"]

    # IMPORTANT:
    # Retrieval functions expect a LIST of queries

    retrieval_queries = [
        original_query
    ]


    print("\n" + "=" * 80)
    print(f"QUESTION {number}")
    print("=" * 80)

    print(original_query)

    print("\nGround truth:")
    print(relevant_indices)


    # =====================================================
    # SEARCH SPACE
    # =====================================================

    # Full corpus.
    # No metadata filtering during evaluation.

    search_corpus = active_corpus

    original_indices = np.arange(
        len(active_corpus)
    )

    search_embeddings = corpus_embeddings

    search_bm25_index = bm25_index


    # =====================================================
    # A. DENSE RETRIEVAL
    # =====================================================

    dense_local_rankings = retrieve_top_k(
        retrieval_queries,
        search_embeddings,
        model,
        top_k=20
    )


    # We only have one query,
    # so take the first ranking.

    dense_local_indices = dense_local_rankings[0]


    dense_indices = [
        int(original_indices[index])
        for index in dense_local_indices
    ]


    # =====================================================
    # B. BM25 RETRIEVAL
    # =====================================================

    bm25_local_rankings = bm25_retrival(
        retrieval_queries,
        search_bm25_index,
        top_k=20
    )


    # Again, one original query only

    bm25_local_indices = bm25_local_rankings[0]


    bm25_indices = [
        int(original_indices[index])
        for index in bm25_local_indices
    ]


    # =====================================================
    # C. HYBRID — DENSE + BM25 + RRF
    # =====================================================

    all_rankings = [
        dense_indices,
        bm25_indices
    ]


    hybrid_results = hybrid_search(
        all_rankings,
        k=60,
        top_k=20
    )


    hybrid_indices = [
        result[0]
        for result in hybrid_results
    ]


    # =====================================================
    # D. HYBRID + RERANKER
    # =====================================================

    reranked_results = reranking(
        original_query,
        active_corpus,
        hybrid_results,
        reranker,
        top_k=5
    )


    reranked_indices = [
        result[0]
        for result in reranked_results
    ]


    # =====================================================
    # DENSE METRICS
    # =====================================================

    dense_hit = hit_at_k(
        dense_indices,
        relevant_indices,
        k=5
    )

    dense_recall = recall_at_k(
        dense_indices,
        relevant_indices,
        k=5
    )

    dense_rr = reciprocal_rank(
        dense_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # BM25 METRICS
    # =====================================================

    bm25_hit = hit_at_k(
        bm25_indices,
        relevant_indices,
        k=5
    )

    bm25_recall = recall_at_k(
        bm25_indices,
        relevant_indices,
        k=5
    )

    bm25_rr = reciprocal_rank(
        bm25_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # HYBRID METRICS
    # =====================================================

    hybrid_hit = hit_at_k(
        hybrid_indices,
        relevant_indices,
        k=5
    )

    hybrid_recall = recall_at_k(
        hybrid_indices,
        relevant_indices,
        k=5
    )

    hybrid_rr = reciprocal_rank(
        hybrid_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # RERANKED METRICS
    # =====================================================

    reranked_hit = hit_at_k(
        reranked_indices,
        relevant_indices,
        k=5
    )

    reranked_recall = recall_at_k(
        reranked_indices,
        relevant_indices,
        k=5
    )

    reranked_rr = reciprocal_rank(
        reranked_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # SAVE RESULTS
    # =====================================================

    ablation_results.append({

        "question": original_query,

        # Dense
        "dense_hit@5": dense_hit,
        "dense_recall@5": dense_recall,
        "dense_rr@5": dense_rr,

        # BM25
        "bm25_hit@5": bm25_hit,
        "bm25_recall@5": bm25_recall,
        "bm25_rr@5": bm25_rr,

        # Hybrid
        "hybrid_hit@5": hybrid_hit,
        "hybrid_recall@5": hybrid_recall,
        "hybrid_rr@5": hybrid_rr,

        # Hybrid + reranker
        "reranked_hit@5": reranked_hit,
        "reranked_recall@5": reranked_recall,
        "reranked_rr@5": reranked_rr
    })


    # =====================================================
    # PRINT PER-QUESTION RESULTS
    # =====================================================

    print("\n----------------------------------------")
    print("DENSE")
    print("----------------------------------------")

    print("Top 5:")
    print(dense_indices[:5])

    print(
        f"Hit@5: {dense_hit} | "
        f"Recall@5: {dense_recall:.3f} | "
        f"RR@5: {dense_rr:.3f}"
    )


    print("\n----------------------------------------")
    print("BM25")
    print("----------------------------------------")

    print("Top 5:")
    print(bm25_indices[:5])

    print(
        f"Hit@5: {bm25_hit} | "
        f"Recall@5: {bm25_recall:.3f} | "
        f"RR@5: {bm25_rr:.3f}"
    )


    print("\n----------------------------------------")
    print("HYBRID — DENSE + BM25 + RRF")
    print("----------------------------------------")

    print("Top 5:")
    print(hybrid_indices[:5])

    print(
        f"Hit@5: {hybrid_hit} | "
        f"Recall@5: {hybrid_recall:.3f} | "
        f"RR@5: {hybrid_rr:.3f}"
    )


    print("\n----------------------------------------")
    print("HYBRID + RERANKER")
    print("----------------------------------------")

    print("Top 5:")
    print(reranked_indices[:5])

    print(
        f"Hit@5: {reranked_hit} | "
        f"Recall@5: {reranked_recall:.3f} | "
        f"RR@5: {reranked_rr:.3f}"
    )


# =========================================================
# AGGREGATE DENSE RESULTS
# =========================================================

dense_hit_rate = np.mean([
    result["dense_hit@5"]
    for result in ablation_results
])

dense_mean_recall = np.mean([
    result["dense_recall@5"]
    for result in ablation_results
])

dense_mrr = np.mean([
    result["dense_rr@5"]
    for result in ablation_results
])


# =========================================================
# AGGREGATE BM25 RESULTS
# =========================================================

bm25_hit_rate = np.mean([
    result["bm25_hit@5"]
    for result in ablation_results
])

bm25_mean_recall = np.mean([
    result["bm25_recall@5"]
    for result in ablation_results
])

bm25_mrr = np.mean([
    result["bm25_rr@5"]
    for result in ablation_results
])


# =========================================================
# AGGREGATE HYBRID RESULTS
# =========================================================

hybrid_hit_rate = np.mean([
    result["hybrid_hit@5"]
    for result in ablation_results
])

hybrid_mean_recall = np.mean([
    result["hybrid_recall@5"]
    for result in ablation_results
])

hybrid_mrr = np.mean([
    result["hybrid_rr@5"]
    for result in ablation_results
])


# =========================================================
# AGGREGATE HYBRID + RERANKER RESULTS
# =========================================================

reranked_hit_rate = np.mean([
    result["reranked_hit@5"]
    for result in ablation_results
])

reranked_mean_recall = np.mean([
    result["reranked_recall@5"]
    for result in ablation_results
])

reranked_mrr = np.mean([
    result["reranked_rr@5"]
    for result in ablation_results
])


# =========================================================
# FINAL ABLATION SUMMARY
# =========================================================

print("\n\n" + "=" * 80)
print("RETRIEVAL ABLATION RESULTS")
print("=" * 80)


print("\nA. DENSE ONLY")
print(f"Hit Rate@5:    {dense_hit_rate:.3f}")
print(f"Mean Recall@5: {dense_mean_recall:.3f}")
print(f"MRR@5:         {dense_mrr:.3f}")


print("\nB. BM25 ONLY")
print(f"Hit Rate@5:    {bm25_hit_rate:.3f}")
print(f"Mean Recall@5: {bm25_mean_recall:.3f}")
print(f"MRR@5:         {bm25_mrr:.3f}")


print("\nC. DENSE + BM25 + RRF")
print(f"Hit Rate@5:    {hybrid_hit_rate:.3f}")
print(f"Mean Recall@5: {hybrid_mean_recall:.3f}")
print(f"MRR@5:         {hybrid_mrr:.3f}")


print("\nD. DENSE + BM25 + RRF + RERANKER")
print(f"Hit Rate@5:    {reranked_hit_rate:.3f}")
print(f"Mean Recall@5: {reranked_mean_recall:.3f}")
print(f"MRR@5:         {reranked_mrr:.3f}")


# =========================================================
# SIMPLE COMPARISON TABLE
# =========================================================

print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

print(
    f"{'SYSTEM':<35}"
    f"{'HIT@5':<12}"
    f"{'RECALL@5':<15}"
    f"{'MRR@5':<12}"
)

print("-" * 74)

print(
    f"{'Dense':<35}"
    f"{dense_hit_rate:<12.3f}"
    f"{dense_mean_recall:<15.3f}"
    f"{dense_mrr:<12.3f}"
)

print(
    f"{'BM25':<35}"
    f"{bm25_hit_rate:<12.3f}"
    f"{bm25_mean_recall:<15.3f}"
    f"{bm25_mrr:<12.3f}"
)

print(
    f"{'Dense + BM25 + RRF':<35}"
    f"{hybrid_hit_rate:<12.3f}"
    f"{hybrid_mean_recall:<15.3f}"
    f"{hybrid_mrr:<12.3f}"
)

print(
    f"{'Dense + BM25 + RRF + Reranker':<35}"
    f"{reranked_hit_rate:<12.3f}"
    f"{reranked_mean_recall:<15.3f}"
    f"{reranked_mrr:<12.3f}"
)


QUESTION 1
Why is masking necessary in the Transformer decoder?

Ground truth:
[12, 19]

----------------------------------------
DENSE
----------------------------------------
Top 5:
[11, 12, 130, 10, 18]
Hit@5: 1 | Recall@5: 0.500 | RR@5: 0.500

----------------------------------------
BM25
----------------------------------------
Top 5:
[19, 16, 201, 122, 39]
Hit@5: 1 | Recall@5: 0.500 | RR@5: 1.000

----------------------------------------
HYBRID — DENSE + BM25 + RRF
----------------------------------------
Top 5:
[19, 11, 18, 23, 85]
Hit@5: 1 | Recall@5: 0.500 | RR@5: 1.000

----------------------------------------
HYBRID + RERANKER
----------------------------------------
Top 5:
[19, 12, 18, 11, 39]
Hit@5: 1 | Recall@5: 1.000 | RR@5: 1.000

QUESTION 2
How does the Transformer decoder generate an output sequence?

Ground truth:
[9, 10]

----------------------------------------
DENSE
----------------------------------------
Top 5:
[10, 11, 12, 18, 9]
Hit@5: 1 | Recall@5: 1.000 | R

In [42]:
# =========================================================
# RERANKER ABLATION
#
# A. Dense only
# B. Dense + Reranker
# C. Dense + BM25 + RRF
# D. Dense + BM25 + RRF + Reranker
#
# Original query only
# =========================================================


reranker_ablation_results = []


for number, example in enumerate(eval_dataset, start=1):

    # Skip unanswerable questions for retrieval metrics
    if example["should_abstain"]:
        continue


    original_query = example["question"]
    relevant_indices = example["relevant_indices"]

    retrieval_queries = [
        original_query
    ]


    print("\n" + "=" * 80)
    print(f"QUESTION {number}")
    print("=" * 80)

    print(original_query)


    # =====================================================
    # SEARCH SPACE
    # =====================================================

    search_embeddings = corpus_embeddings

    original_indices = np.arange(
        len(active_corpus)
    )

    search_bm25_index = bm25_index


    # =====================================================
    # A. DENSE RETRIEVAL
    # =====================================================

    dense_local_rankings = retrieve_top_k(
        retrieval_queries,
        search_embeddings,
        model,
        top_k=20
    )

    dense_local_indices = dense_local_rankings[0]

    dense_indices = [
        int(original_indices[index])
        for index in dense_local_indices
    ]


    # =====================================================
    # B. DENSE + RERANKER
    # =====================================================

    # reranking() expects candidates in the same format
    # as hybrid_results:
    #
    # (chunk_index, score)
    #
    # Here we only care about supplying the candidate indices.
    # The cross-encoder will calculate its own relevance scores.

    dense_candidates = [
        (index, 0.0)
        for index in dense_indices[:20]
    ]


    dense_reranked_results = reranking(
        original_query,
        active_corpus,
        dense_candidates,
        reranker,
        top_k=5
    )


    dense_reranked_indices = [
        result[0]
        for result in dense_reranked_results
    ]


    # =====================================================
    # BM25 RETRIEVAL
    # =====================================================

    bm25_local_rankings = bm25_retrival(
        retrieval_queries,
        search_bm25_index,
        top_k=20
    )

    bm25_local_indices = bm25_local_rankings[0]

    bm25_indices = [
        int(original_indices[index])
        for index in bm25_local_indices
    ]


    # =====================================================
    # C. HYBRID — DENSE + BM25 + RRF
    # =====================================================

    all_rankings = [
        dense_indices,
        bm25_indices
    ]


    hybrid_results = hybrid_search(
        all_rankings,
        k=60,
        top_k=20
    )


    hybrid_indices = [
        result[0]
        for result in hybrid_results
    ]


    # =====================================================
    # D. HYBRID + RERANKER
    # =====================================================

    hybrid_reranked_results = reranking(
        original_query,
        active_corpus,
        hybrid_results,
        reranker,
        top_k=5
    )


    hybrid_reranked_indices = [
        result[0]
        for result in hybrid_reranked_results
    ]


    # =====================================================
    # A. DENSE METRICS
    # =====================================================

    dense_hit = hit_at_k(
        dense_indices,
        relevant_indices,
        k=5
    )

    dense_recall = recall_at_k(
        dense_indices,
        relevant_indices,
        k=5
    )

    dense_rr = reciprocal_rank(
        dense_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # B. DENSE + RERANKER METRICS
    # =====================================================

    dense_reranked_hit = hit_at_k(
        dense_reranked_indices,
        relevant_indices,
        k=5
    )

    dense_reranked_recall = recall_at_k(
        dense_reranked_indices,
        relevant_indices,
        k=5
    )

    dense_reranked_rr = reciprocal_rank(
        dense_reranked_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # C. HYBRID METRICS
    # =====================================================

    hybrid_hit = hit_at_k(
        hybrid_indices,
        relevant_indices,
        k=5
    )

    hybrid_recall = recall_at_k(
        hybrid_indices,
        relevant_indices,
        k=5
    )

    hybrid_rr = reciprocal_rank(
        hybrid_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # D. HYBRID + RERANKER METRICS
    # =====================================================

    hybrid_reranked_hit = hit_at_k(
        hybrid_reranked_indices,
        relevant_indices,
        k=5
    )

    hybrid_reranked_recall = recall_at_k(
        hybrid_reranked_indices,
        relevant_indices,
        k=5
    )

    hybrid_reranked_rr = reciprocal_rank(
        hybrid_reranked_indices,
        relevant_indices,
        k=5
    )


    # =====================================================
    # SAVE RESULTS
    # =====================================================

    reranker_ablation_results.append({

        "question": original_query,

        "dense_hit": dense_hit,
        "dense_recall": dense_recall,
        "dense_rr": dense_rr,

        "dense_reranked_hit": dense_reranked_hit,
        "dense_reranked_recall": dense_reranked_recall,
        "dense_reranked_rr": dense_reranked_rr,

        "hybrid_hit": hybrid_hit,
        "hybrid_recall": hybrid_recall,
        "hybrid_rr": hybrid_rr,

        "hybrid_reranked_hit": hybrid_reranked_hit,
        "hybrid_reranked_recall": hybrid_reranked_recall,
        "hybrid_reranked_rr": hybrid_reranked_rr
    })


    # =====================================================
    # PRINT PER QUESTION
    # =====================================================

    print("\nGround truth:")
    print(relevant_indices)


    print("\nDENSE")
    print(dense_indices[:5])

    print(
        f"Hit: {dense_hit} | "
        f"Recall: {dense_recall:.3f} | "
        f"RR: {dense_rr:.3f}"
    )


    print("\nDENSE + RERANKER")
    print(dense_reranked_indices)

    print(
        f"Hit: {dense_reranked_hit} | "
        f"Recall: {dense_reranked_recall:.3f} | "
        f"RR: {dense_reranked_rr:.3f}"
    )


    print("\nHYBRID")
    print(hybrid_indices[:5])

    print(
        f"Hit: {hybrid_hit} | "
        f"Recall: {hybrid_recall:.3f} | "
        f"RR: {hybrid_rr:.3f}"
    )


    print("\nHYBRID + RERANKER")
    print(hybrid_reranked_indices)

    print(
        f"Hit: {hybrid_reranked_hit} | "
        f"Recall: {hybrid_reranked_recall:.3f} | "
        f"RR: {hybrid_reranked_rr:.3f}"
    )


# =========================================================
# AGGREGATE FUNCTION
# =========================================================

def average_metric(results, key):

    return np.mean([
        result[key]
        for result in results
    ])


# =========================================================
# A. DENSE
# =========================================================

dense_hit_rate = average_metric(
    reranker_ablation_results,
    "dense_hit"
)

dense_recall = average_metric(
    reranker_ablation_results,
    "dense_recall"
)

dense_mrr = average_metric(
    reranker_ablation_results,
    "dense_rr"
)


# =========================================================
# B. DENSE + RERANKER
# =========================================================

dense_reranked_hit_rate = average_metric(
    reranker_ablation_results,
    "dense_reranked_hit"
)

dense_reranked_recall = average_metric(
    reranker_ablation_results,
    "dense_reranked_recall"
)

dense_reranked_mrr = average_metric(
    reranker_ablation_results,
    "dense_reranked_rr"
)


# =========================================================
# C. HYBRID
# =========================================================

hybrid_hit_rate = average_metric(
    reranker_ablation_results,
    "hybrid_hit"
)

hybrid_recall = average_metric(
    reranker_ablation_results,
    "hybrid_recall"
)

hybrid_mrr = average_metric(
    reranker_ablation_results,
    "hybrid_rr"
)


# =========================================================
# D. HYBRID + RERANKER
# =========================================================

hybrid_reranked_hit_rate = average_metric(
    reranker_ablation_results,
    "hybrid_reranked_hit"
)

hybrid_reranked_recall = average_metric(
    reranker_ablation_results,
    "hybrid_reranked_recall"
)

hybrid_reranked_mrr = average_metric(
    reranker_ablation_results,
    "hybrid_reranked_rr"
)


# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n\n" + "=" * 80)
print("RERANKER ABLATION SUMMARY")
print("=" * 80)


print(
    f"{'SYSTEM':<38}"
    f"{'HIT@5':<12}"
    f"{'RECALL@5':<15}"
    f"{'MRR@5':<12}"
)

print("-" * 77)


print(
    f"{'Dense':<38}"
    f"{dense_hit_rate:<12.3f}"
    f"{dense_recall:<15.3f}"
    f"{dense_mrr:<12.3f}"
)


print(
    f"{'Dense + Reranker':<38}"
    f"{dense_reranked_hit_rate:<12.3f}"
    f"{dense_reranked_recall:<15.3f}"
    f"{dense_reranked_mrr:<12.3f}"
)


print(
    f"{'Dense + BM25 + RRF':<38}"
    f"{hybrid_hit_rate:<12.3f}"
    f"{hybrid_recall:<15.3f}"
    f"{hybrid_mrr:<12.3f}"
)


print(
    f"{'Dense + BM25 + RRF + Reranker':<38}"
    f"{hybrid_reranked_hit_rate:<12.3f}"
    f"{hybrid_reranked_recall:<15.3f}"
    f"{hybrid_reranked_mrr:<12.3f}"
)


QUESTION 1
Why is masking necessary in the Transformer decoder?

Ground truth:
[12, 19]

DENSE
[11, 12, 130, 10, 18]
Hit: 1 | Recall: 0.500 | RR: 0.500

DENSE + RERANKER
[19, 12, 18, 206, 11]
Hit: 1 | Recall: 1.000 | RR: 1.000

HYBRID
[19, 11, 18, 23, 85]
Hit: 1 | Recall: 0.500 | RR: 1.000

HYBRID + RERANKER
[19, 12, 18, 11, 39]
Hit: 1 | Recall: 1.000 | RR: 1.000

QUESTION 2
How does the Transformer decoder generate an output sequence?

Ground truth:
[9, 10]

DENSE
[10, 11, 12, 18, 9]
Hit: 1 | Recall: 1.000 | RR: 1.000

DENSE + RERANKER
[10, 9, 18, 20, 12]
Hit: 1 | Recall: 1.000 | RR: 1.000

HYBRID
[12, 10, 11, 9, 18]
Hit: 1 | Recall: 1.000 | RR: 0.500

HYBRID + RERANKER
[10, 9, 18, 20, 12]
Hit: 1 | Recall: 1.000 | RR: 1.000

QUESTION 3
How does the Transformer decoder architecture differ from the encoder?

Ground truth:
[11, 12]

DENSE
[11, 18, 12, 34, 10]
Hit: 1 | Recall: 1.000 | RR: 1.000

DENSE + RERANKER
[11, 18, 9, 12, 39]
Hit: 1 | Recall: 1.000 | RR: 1.000

HYBRID
[11, 18, 12, 

In [43]:
#Completeness

# =========================================================
# ANSWER QUALITY TEST — QUESTION 1
# =========================================================

example = eval_dataset[0]

original_query = example["question"]
relevant_indices = example["relevant_indices"]
expected_facts = example["expected_facts"]

retrieval_queries = [original_query]


# =========================================================
# 1. DENSE RETRIEVAL
# =========================================================

dense_local_rankings = retrieve_top_k(
    retrieval_queries,
    corpus_embeddings,
    model,
    top_k=20
)

dense_indices = [
    int(index)
    for index in dense_local_rankings[0]
]


# =========================================================
# 2. PREPARE DENSE CANDIDATES FOR RERANKER
# =========================================================

dense_candidates = [
    (index, 0.0)
    for index in dense_indices
]


# =========================================================
# 3. RERANK
# =========================================================

reranked_results = reranking(
    original_query,
    active_corpus,
    dense_candidates,
    reranker,
    top_k=5
)

final_indices = [
    result[0]
    for result in reranked_results
]


# =========================================================
# 4. CREATE CONTEXT
# =========================================================

final_context = context_from_indices(
    active_corpus,
    final_indices
)


# =========================================================
# 5. GENERATE ANSWER
# =========================================================

answer = generation(
    original_query,
    final_context,
    generation_model,
    tokenizer
)


# =========================================================
# 6. PRINT EVERYTHING
# =========================================================

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(original_query)

print("\n" + "=" * 80)
print("EXPECTED FACTS")
print("=" * 80)

for i, fact in enumerate(expected_facts, start=1):
    print(f"{i}. {fact}")


print("\n" + "=" * 80)
print("RERANKED TOP 5 INDICES")
print("=" * 80)
print(final_indices)


print("\n" + "=" * 80)
print("RETRIEVED CONTEXT")
print("=" * 80)

for i, item in enumerate(final_context, start=1):

    print(f"\n--- Context {i} ---")
    print("Document:", item["document"])
    print("Page:", item["page"])
    print(item["text"])


print("\n" + "=" * 80)
print("GENERATED ANSWER")
print("=" * 80)
print(answer)

QUESTION
Why is masking necessary in the Transformer decoder?

EXPECTED FACTS
1. decoder positions must not attend to future positions
2. masking preserves autoregressive generation
3. position i depends only on known earlier outputs

RERANKED TOP 5 INDICES
[19, 12, 18, 206, 11]

RETRIEVED CONTEXT

--- Context 1 ---
Document: /content/1706.03762v7.pdf
Page: 5
Each position in the encoder can attend to all positions in the previous layer of the
encoder. • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
information flow in the decoder to preserve the auto-regressive property. We implement this
inside of scaled dot-product attention by masking out (setting to −∞) all values in the input
of the softmax which correspond to illegal connections. See Figure 2. 3.3 Position-wise Feed-Forward Networks
In addition to attention sub-layers, each of the layers i

In [44]:
# =========================================================
# ABSTENTION TEST — UNANSWERABLE QUESTION 1
# =========================================================

example = eval_dataset[10]

original_query = example["question"]
should_abstain = example["should_abstain"]

retrieval_queries = [original_query]


# =========================================================
# 1. PRINT QUESTION
# =========================================================

print("=" * 80)
print("QUESTION")
print("=" * 80)

print(original_query)

print("\nExpected abstention:")
print(should_abstain)


# =========================================================
# 2. DENSE RETRIEVAL
# =========================================================

dense_local_rankings = retrieve_top_k(
    retrieval_queries,
    corpus_embeddings,
    model,
    top_k=20
)

dense_indices = [
    int(index)
    for index in dense_local_rankings[0]
]


# =========================================================
# 3. PREPARE CANDIDATES FOR RERANKER
# =========================================================

dense_candidates = [
    (index, 0.0)
    for index in dense_indices
]


# =========================================================
# 4. RERANK
# =========================================================

reranked_results = reranking(
    original_query,
    active_corpus,
    dense_candidates,
    reranker,
    top_k=5
)

final_indices = [
    result[0]
    for result in reranked_results
]


# =========================================================
# 5. CREATE FINAL CONTEXT
# =========================================================

final_context = context_from_indices(
    active_corpus,
    final_indices
)


# =========================================================
# 6. GENERATE ANSWER
# =========================================================

answer = generation(
    original_query,
    final_context,
    generation_model,
    tokenizer
)


# =========================================================
# 7. PRINT RETRIEVAL RESULTS
# =========================================================

print("\n" + "=" * 80)
print("DENSE TOP 20")
print("=" * 80)

print(dense_indices)


print("\n" + "=" * 80)
print("RERANKED TOP 5")
print("=" * 80)

print(final_indices)


# =========================================================
# 8. PRINT RETRIEVED CONTEXT
# =========================================================

print("\n" + "=" * 80)
print("RETRIEVED CONTEXT")
print("=" * 80)

for i, item in enumerate(final_context, start=1):

    print(f"\n--- Context {i} ---")

    print(
        "Document:",
        item["document"]
    )

    print(
        "Page:",
        item["page"]
    )

    print(
        item["text"]
    )


# =========================================================
# 9. PRINT GENERATED ANSWER
# =========================================================

print("\n" + "=" * 80)
print("GENERATED ANSWER")
print("=" * 80)

print(answer)

QUESTION
What was the exact financial cost in US dollars of training the Transformer big model?

Expected abstention:
True

DENSE TOP 20
[33, 31, 30, 32, 34, 130, 1, 94, 11, 97, 40, 37, 2, 39, 29, 18, 92, 154, 215, 98]

RERANKED TOP 5
[32, 31, 30, 1, 33]

RETRIEVED CONTEXT

--- Context 1 ---
Document: /content/1706.03762v7.pdf
Page: 8
Even our base model
surpasses all previously published models and ensembles, at a fraction of the training cost of any of
the competitive models. On the WMT 2014 English-to-French translation task, our big model achieves a BLEU score of 41.0,
outperforming all of the previously published single models, at less than 1/4 the training cost of the
previous state-of-the-art model. The Transformer (big) model trained for English-to-French used
dropout rate Pdrop = 0.1, instead of 0.3. For the base models, we used a single model obtained by averaging the last 5 checkpoints, which
were written at 10-minute intervals. For the big models, we averaged the last 20 ch

In [45]:
# =========================================================
# GENERATE ANSWERS FOR FULL EVALUATION DATASET
#
# Current default pipeline:
# Query
# → Dense Top-20
# → Reranker Top-5
# → Context
# → Generation
# =========================================================

answer_evaluation_results = []


for number, example in enumerate(eval_dataset, start=1):

    original_query = example["question"]
    expected_facts = example["expected_facts"]
    should_abstain = example["should_abstain"]

    retrieval_queries = [original_query]


    print("\n" + "=" * 80)
    print(f"QUESTION {number}/{len(eval_dataset)}")
    print("=" * 80)
    print(original_query)


    # =====================================================
    # 1. DENSE RETRIEVAL — TOP 20
    # =====================================================

    dense_local_rankings = retrieve_top_k(
        retrieval_queries,
        corpus_embeddings,
        model,
        top_k=20
    )

    dense_indices = [
        int(index)
        for index in dense_local_rankings[0]
    ]


    # =====================================================
    # 2. PREPARE CANDIDATES FOR RERANKER
    # =====================================================

    dense_candidates = [
        (index, 0.0)
        for index in dense_indices
    ]


    # =====================================================
    # 3. RERANK — TOP 5
    # =====================================================

    reranked_results = reranking(
        original_query,
        active_corpus,
        dense_candidates,
        reranker,
        top_k=5
    )

    final_indices = [
        result[0]
        for result in reranked_results
    ]


    # =====================================================
    # 4. BUILD FINAL CONTEXT
    # =====================================================

    final_context = context_from_indices(
        active_corpus,
        final_indices
    )


    # =====================================================
    # 5. GENERATE ANSWER
    # =====================================================

    answer = generation(
        original_query,
        final_context,
        generation_model,
        tokenizer
    )


    # =====================================================
    # 6. SAVE EVERYTHING
    # =====================================================

    answer_evaluation_results.append({

        "question_number": number,

        "question": original_query,

        "expected_facts": expected_facts,

        "should_abstain": should_abstain,

        "dense_top20": dense_indices,

        "reranked_top5": final_indices,

        "context": final_context,

        "answer": answer
    })


    # =====================================================
    # 7. PRINT SHORT RESULT
    # =====================================================

    print("\nReranked Top 5:")
    print(final_indices)

    print("\nExpected abstention:")
    print(should_abstain)

    print("\nGenerated answer:")
    print(answer)


# =========================================================
# FINISHED
# =========================================================

print("\n\n" + "=" * 80)
print("ANSWER GENERATION COMPLETE")
print("=" * 80)

print(
    "Questions evaluated:",
    len(answer_evaluation_results)
)


QUESTION 1/12
Why is masking necessary in the Transformer decoder?

Reranked Top 5:
[19, 12, 18, 206, 11]

Expected abstention:
False

Generated answer:
Masking is necessary in the Transformer decoder because it prevents leftward information flow, preserving the auto-regressive property. Specifically, the text states that "We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections." This ensures that predictions for position i can depend only on the known outputs at positions less than i.

QUESTION 2/12
How does the Transformer decoder generate an output sequence?

Reranked Top 5:
[10, 9, 18, 20, 12]

Expected abstention:
False

Generated answer:
The Transformer decoder generates an output sequence by taking into account the previously generated symbols through self-attention. Specifically, it consumes the output of the encoder's previous layer as part of the input during each s

In [46]:
for result in answer_evaluation_results:

    print("\n" + "=" * 80)

    print(
        f"QUESTION {result['question_number']}"
    )

    print("=" * 80)

    print(result["question"])

    print("\nExpected facts:")

    if result["expected_facts"]:

        for fact in result["expected_facts"]:
            print("-", fact)

    else:
        print("None — unanswerable question")

    print(
        "\nShould abstain:",
        result["should_abstain"]
    )

    print("\nGenerated answer:")
    print(result["answer"])


QUESTION 1
Why is masking necessary in the Transformer decoder?

Expected facts:
- decoder positions must not attend to future positions
- masking preserves autoregressive generation
- position i depends only on known earlier outputs

Should abstain: False

Generated answer:
Masking is necessary in the Transformer decoder because it prevents leftward information flow, preserving the auto-regressive property. Specifically, the text states that "We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections." This ensures that predictions for position i can depend only on the known outputs at positions less than i.

QUESTION 2
How does the Transformer decoder generate an output sequence?

Expected facts:
- decoder generates symbols one element at a time
- generation is autoregressive
- previously generated symbols are used to generate the next

Should abstain: False

Generated answer:

In [48]:
# =========================================================
# CREATE FAITHFULNESS REVIEW REPORT
# =========================================================

report_lines = []

for result in answer_evaluation_results:

    # Faithfulness applies here to answerable questions
    if result["should_abstain"]:
        continue

    report_lines.append("=" * 80)
    report_lines.append(
        f"QUESTION {result['question_number']}"
    )
    report_lines.append("=" * 80)

    report_lines.append(result["question"])

    report_lines.append("\nEXPECTED FACTS:")

    for fact in result["expected_facts"]:
        report_lines.append(
            f"- {fact}"
        )

    report_lines.append("\nRETRIEVED CONTEXT:")

    for i, item in enumerate(
        result["context"],
        start=1
    ):

        report_lines.append(
            f"\n--- Context {i} ---"
        )

        report_lines.append(
            f"Document: {item['document']}"
        )

        report_lines.append(
            f"Page: {item['page']}"
        )

        report_lines.append(
            item["text"]
        )

    report_lines.append(
        "\nGENERATED ANSWER:"
    )

    report_lines.append(
        result["answer"]
    )

    report_lines.append("\n\n")


# =========================================================
# SAVE REPORT
# =========================================================

faithfulness_report = "\n".join(
    report_lines
)

with open(
    "/content/faithfulness_report.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(faithfulness_report)


print(
    "Saved:",
    "/content/faithfulness_report.txt"
)

Saved: /content/faithfulness_report.txt


In [51]:
# =========================================================
# GENERATE ANSWERS FOR FULL EVALUATION DATASET — V2
#
# Same retrieval pipeline:
# Query
# → Dense Top-20
# → Reranker Top-5
# → Context
#
# Only change:
# generation_v2() instead of generation()
# =========================================================

answer_evaluation_results_v2 = []


for number, example in enumerate(eval_dataset, start=1):

    original_query = example["question"]
    expected_facts = example["expected_facts"]
    should_abstain = example["should_abstain"]

    retrieval_queries = [original_query]


    print("\n" + "=" * 80)
    print(f"QUESTION {number}/{len(eval_dataset)}")
    print("=" * 80)
    print(original_query)


    # =====================================================
    # 1. DENSE RETRIEVAL — TOP 20
    # =====================================================

    dense_local_rankings = retrieve_top_k(
        retrieval_queries,
        corpus_embeddings,
        model,
        top_k=20
    )

    dense_indices = [
        int(index)
        for index in dense_local_rankings[0]
    ]


    # =====================================================
    # 2. PREPARE CANDIDATES FOR RERANKER
    # =====================================================

    dense_candidates = [
        (index, 0.0)
        for index in dense_indices
    ]


    # =====================================================
    # 3. RERANK — TOP 5
    # =====================================================

    reranked_results = reranking(
        original_query,
        active_corpus,
        dense_candidates,
        reranker,
        top_k=5
    )

    final_indices = [
        result[0]
        for result in reranked_results
    ]


    # =====================================================
    # 4. BUILD FINAL CONTEXT
    # =====================================================

    final_context = context_from_indices(
        active_corpus,
        final_indices
    )


    # =====================================================
    # 5. GENERATE ANSWER USING NEW PROMPT
    # =====================================================

    answer = generation(
        original_query,
        final_context,
        generation_model,
        tokenizer
    )


    # =====================================================
    # 6. SAVE RESULTS
    # =====================================================

    answer_evaluation_results_v2.append({

        "question_number": number,

        "question": original_query,

        "expected_facts": expected_facts,

        "should_abstain": should_abstain,

        "dense_top20": dense_indices,

        "reranked_top5": final_indices,

        "context": final_context,

        "answer": answer
    })


    # =====================================================
    # 7. PRINT SHORT RESULT
    # =====================================================

    print("\nReranked Top 5:")
    print(final_indices)

    print("\nExpected abstention:")
    print(should_abstain)

    print("\nGenerated answer:")
    print(answer)


# =========================================================
# FINISHED
# =========================================================

print("\n\n" + "=" * 80)
print("V2 ANSWER GENERATION COMPLETE")
print("=" * 80)

print(
    "Questions evaluated:",
    len(answer_evaluation_results_v2)
)


QUESTION 1/12
Why is masking necessary in the Transformer decoder?

Reranked Top 5:
[19, 12, 18, 206, 11]

Expected abstention:
False

Generated answer:
Masking is necessary in the Transformer decoder because it prevents leftward information flow, preserving the auto-regressive property. Specifically, the text states: "We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections." This ensures that predictions for position i can depend only on the known outputs at positions less than i.

QUESTION 2/12
How does the Transformer decoder generate an output sequence?

Reranked Top 5:
[10, 9, 18, 20, 12]

Expected abstention:
False

Generated answer:
The Transformer decoder generates an output sequence by taking the previously generated symbols as additional input during each step. Specifically, given the encoded representation \( z \), the decoder processes these symbols sequentially t

In [55]:
# =========================================================
# GENERATION MODEL ABLATION — QWEN 7B
#
# SAME:
# - Questions
# - Dense retrieval
# - Top 20
# - Reranker
# - Top 5 context
# - V2 prompt
#
# ONLY CHANGE:
# Qwen2.5-1.5B → Qwen2.5-7B
# =========================================================

answer_evaluation_results_v3 = []


for result in answer_evaluation_results_v2:

    question = result["question"]
    context = result["context"]


    print("\n" + "=" * 80)
    print(
        f"QUESTION {result['question_number']}/"
        f"{len(answer_evaluation_results_v2)}"
    )
    print("=" * 80)

    print(question)


    # =====================================================
    # GENERATE USING STRONGER MODEL
    # =====================================================

    answer = generation(
        question,
        context,
        generation_model_v3,
        tokenizer_v3
    )


    # =====================================================
    # SAVE RESULT
    # =====================================================

    answer_evaluation_results_v3.append({

        "question_number":
            result["question_number"],

        "question":
            question,

        "expected_facts":
            result["expected_facts"],

        "should_abstain":
            result["should_abstain"],

        # exact same retrieval results
        "dense_top20":
            result["dense_top20"],

        "reranked_top5":
            result["reranked_top5"],

        # exact same context
        "context":
            context,

        # new answer only
        "answer":
            answer
    })


    print("\nGenerated answer:")
    print(answer)


print("\n" + "=" * 80)
print("V3 — 7B GENERATION COMPLETE")
print("=" * 80)

print(
    "Questions evaluated:",
    len(answer_evaluation_results_v3)
)


QUESTION 1/12
Why is masking necessary in the Transformer decoder?

Generated answer:
To preserve the auto-regressive property, masking is necessary in the Transformer decoder to prevent leftward information flow, ensuring that predictions for position i can depend only on the known outputs at positions less than i.

QUESTION 2/12
How does the Transformer decoder generate an output sequence?

Generated answer:
The Transformer decoder generates an output sequence (y1, ..., ym) of symbols one element at a time, consuming the previously generated symbols as additional input when generating the next. Additionally, the decoder includes a third sub-layer that performs multi-head attention over the output of the encoder stack. This process is repeated for each position in the output sequence.

QUESTION 3/12
How does the Transformer decoder architecture differ from the encoder?

Generated answer:
The Transformer decoder architecture differs from the encoder in that it includes a third sub-lay

In [56]:
for old, new in zip(
    answer_evaluation_results_v2,
    answer_evaluation_results_v3
):

    print("\n" + "=" * 80)
    print(
        f"QUESTION {old['question_number']}"
    )
    print("=" * 80)

    print(old["question"])

    print("\n--- 1.5B ANSWER ---")
    print(old["answer"])

    print("\n--- 7B ANSWER ---")
    print(new["answer"])


QUESTION 1
Why is masking necessary in the Transformer decoder?

--- 1.5B ANSWER ---
Masking is necessary in the Transformer decoder because it prevents leftward information flow, preserving the auto-regressive property. Specifically, the text states: "We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections." This ensures that predictions for position i can depend only on the known outputs at positions less than i.

--- 7B ANSWER ---
To preserve the auto-regressive property, masking is necessary in the Transformer decoder to prevent leftward information flow, ensuring that predictions for position i can depend only on the known outputs at positions less than i.

QUESTION 2
How does the Transformer decoder generate an output sequence?

--- 1.5B ANSWER ---
The Transformer decoder generates an output sequence by taking the previously generated symbols as additional input during 

In [57]:
import pandas as pd


# =========================================================
# GENERATION ABLATION SUMMARY
# =========================================================

generation_evaluation_summary = pd.DataFrame([
    {
        "version": "V1",
        "generator": "Qwen2.5-1.5B-Instruct",
        "prompt": "Original",
        "completeness": 0.675,
        "faithfulness": 0.81,
        "correctness": 0.75,
        "abstention": 1.00
    },
    {
        "version": "V2",
        "generator": "Qwen2.5-1.5B-Instruct",
        "prompt": "Improved",
        "completeness": 0.71,
        "faithfulness": 0.84,
        "correctness": 0.83,
        "abstention": 1.00
    },
    {
        "version": "V3",
        "generator": "Qwen2.5-7B-Instruct",
        "prompt": "Improved",
        "completeness": 0.83,
        "faithfulness": 0.97,
        "correctness": 0.95,
        "abstention": 1.00
    }
])

print("GENERATION EVALUATION")
display(generation_evaluation_summary)


# =========================================================
# FINAL CURRENT RAG BASELINE
# =========================================================

final_rag_baseline = pd.DataFrame([
    {
        "retrieval_pipeline": "Dense Top20 -> Reranker Top5",
        "retrieval_hit@5": 0.900,
        "retrieval_recall@5": 0.717,
        "retrieval_mrr@5": 0.783,

        "generator": "Qwen2.5-7B-Instruct",
        "generation_prompt": "V2",

        "answer_completeness": 0.83,
        "answer_faithfulness": 0.97,
        "answer_correctness": 0.95,
        "abstention": 1.00
    }
])

print("\nFINAL CURRENT RAG BASELINE")
display(final_rag_baseline)


# =========================================================
# SAVE TO CSV
# =========================================================

generation_evaluation_summary.to_csv(
    "/content/generation_evaluation_summary.csv",
    index=False
)

final_rag_baseline.to_csv(
    "/content/final_rag_baseline.csv",
    index=False
)

print("\nSaved:")
print("/content/generation_evaluation_summary.csv")
print("/content/final_rag_baseline.csv")

GENERATION EVALUATION


,version,generator,prompt,completeness,faithfulness,correctness,abstention
0,V1,Qwen2.5-1.5B-Instruct,Original,0.675,0.81,0.75,1.0
1,V2,Qwen2.5-1.5B-Instruct,Improved,0.710,0.84,0.83,1.0
2,V3,Qwen2.5-7B-Instruct,Improved,0.830,0.97,0.95,1.0



FINAL CURRENT RAG BASELINE


,retrieval_pipeline,retrieval_hit@5,retrieval_recall@5,retrieval_mrr@5,generator,generation_prompt,answer_completeness,answer_faithfulness,answer_correctness,abstention
0,Dense Top20 -> Reranker Top5,0.9,0.717,0.783,Qwen2.5-7B-Instruct,V2,0.83,0.97,0.95,1.0



Saved:
/content/generation_evaluation_summary.csv
/content/final_rag_baseline.csv
